In [0]:
from __future__ import annotations

# ── Imports ──────────────────────────────────────────────────────────────────
import json
import logging
import re
import threading
import time
from functools import reduce
from typing import List, Optional

from delta.tables import DeltaTable
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

class _EtlJsonFormatter(logging.Formatter):
    """Emit one compact JSON line per log record.

    Every record always contains: timestamp, level, pipeline, table,
    scd_type, batch_date, stage, and message.  Extra kwargs passed to
    logger.info(..., extra={...}) are merged at the top level.
    """

    PIPELINE_NAME: str = "f1_silver_etl"

    def format(self, record: logging.LogRecord) -> str:
        """Serialize *record* to a single-line JSON string.

        Args:
            record: The :class:`logging.LogRecord` to format.

        Returns:
            JSON string with all standard ETL fields plus any extras.
        """
        payload: dict = {
            "timestamp": self.formatTime(record, "%Y-%m-%dT%H:%M:%S"),
            "level": record.levelname,
            "pipeline": self.PIPELINE_NAME,
            "table": getattr(record, "table", ""),
            "scd_type": getattr(record, "scd_type", ""),
            "batch_date": getattr(record, "batch_date", ""),
            "stage": getattr(record, "stage", ""),
            "message": record.getMessage(),
        }
        # Merge any ad-hoc extra fields (elapsed_s, counts, etc.)
        for key, val in record.__dict__.items():
            if key not in (
                "msg", "args", "levelname", "levelno", "pathname", "filename",
                "module", "exc_info", "exc_text", "stack_info", "lineno",
                "funcName", "created", "msecs", "relativeCreated", "thread",
                "threadName", "processName", "process", "name", "message",
                "asctime",
            ) and not key.startswith("_"):
                payload.setdefault(key, val)
        return json.dumps(payload, default=str)


def _build_logger(name: str = "etl") -> logging.Logger:
    """Create and return the structured JSON logger.

    Args:
        name: Logger namespace.

    Returns:
        Configured :class:`logging.Logger` instance.
    """
    logger = logging.getLogger(name)
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(_EtlJsonFormatter())
        logger.addHandler(handler)
    logger.setLevel(logging.DEBUG)
    logger.propagate = False
    return logger


# Module-level logger; _LogCtx enriches every subsequent record.
_LOG: logging.Logger = _build_logger()


class _LogCtx:
    """Context manager that injects table/scd_type/batch_date into log records.

    Uses ``threading.local()`` so concurrent notebook tasks on the same
    Python interpreter never bleed context into each other.

    Usage::

        with _LogCtx(table="drivers", scd_type="SCD2", batch_date="2024-01-01"):
            _LOG.info("stage started", extra={"stage": "load"})
    """

    _tls: threading.local = threading.local()

    @classmethod
    def _state(cls) -> dict:
        """Return the thread-local context dict, initialising it if absent.

        Returns:
            Mutable dict of context fields for the current thread.
        """
        if not hasattr(cls._tls, "current"):
            cls._tls.current = {}
        return cls._tls.current

    def __init__(self, **kwargs: str) -> None:
        """Initialise the context manager with fields to inject.

        Args:
            **kwargs: Key-value pairs (e.g. ``table="drivers"``) that will be
                set as attributes on every :class:`logging.LogRecord` emitted
                while this context is active.
        """
        self._fields = kwargs
        self._prev: dict = {}

    def __enter__(self) -> "_LogCtx":
        """Activate the context: push current state and install the filter.

        Returns:
            This :class:`_LogCtx` instance.
        """
        self._prev = dict(_LogCtx._state())
        _LogCtx._state().update(self._fields)
        _LOG.handlers[0].addFilter(self)  # type: ignore[arg-type]
        return self

    def filter(self, record: logging.LogRecord) -> bool:
        """Inject thread-local context fields into *record* before emission.

        Args:
            record: Log record being processed.

        Returns:
            Always ``True`` — this filter enriches but never suppresses records.
        """
        for k, v in _LogCtx._state().items():
            setattr(record, k, v)
        return True

    def __exit__(self, *_: object) -> None:
        """Deactivate the context: restore previous state and remove the filter.

        Args:
            *_: Exception info (ignored — exceptions propagate normally).
        """
        _LogCtx._tls.current = self._prev
        try:
            _LOG.handlers[0].removeFilter(self)  # type: ignore[arg-type]
        except ValueError:
            pass


# ── Spark Session ─────────────────────────────────────────────────────────────
spark: SparkSession = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")


# ── Widget Declarations & Parameters ─────────────────────────────────────────

# Declare widgets first so Databricks renders the parameter UI and so that
# .get() never throws an "unknown widget" error on a fresh run.
dbutils.widgets.text("table_name", "", "Table Name (e.g. drivers)")   # type: ignore[name-defined]  # noqa: F821
dbutils.widgets.text("table_type", "", "Table Type: DIM or FACT")     # type: ignore[name-defined]  # noqa: F821
dbutils.widgets.text("batch_date", "", "Batch Date (yyyy-MM-dd)")     # type: ignore[name-defined]  # noqa: F821

_SAFE_TABLE_NAME_RE: re.Pattern = re.compile(r"^[A-Za-z_][A-Za-z0-9_]{0,127}$")
_SAFE_DATE_RE: re.Pattern       = re.compile(r"^\d{4}-\d{2}-\d{2}$")
# FIX-04: TABLE_TYPE now validated with an explicit regex, consistent with
# TABLE_NAME and BATCH_DATE.  Prevents any unexpected value reaching SQL.
_SAFE_TABLE_TYPE_RE: re.Pattern = re.compile(r"^(DIM|FACT)$")


def _sanitise_param(name: str, value: str, pattern: re.Pattern) -> str:
    """Validate a widget value against *pattern* before it enters any SQL string.

    Args:
        name: Widget name (used in error messages).
        value: Raw widget value.
        pattern: Compiled regex the value must fully match.

    Returns:
        The original *value* if it matches.

    Raises:
        ValueError: If *value* does not match *pattern*.
    """
    if not pattern.fullmatch(value):
        raise ValueError(
            f"Widget parameter '{name}' value {value!r} failed safety check "
            f"(pattern: {pattern.pattern}). Possible injection attempt."
        )
    return value


def _get_param(name: str) -> str:
    """Read a Databricks widget parameter or raise a clear error.

    Args:
        name: Widget / task parameter name.

    Returns:
        Non-empty string value of the parameter.

    Raises:
        ValueError: If the parameter is missing or blank.
    """
    try:
        val: str = dbutils.widgets.get(name).strip()  # type: ignore[name-defined]  # noqa: F821
    except Exception as exc:
        raise ValueError(f"Widget parameter '{name}' is missing: {exc}") from exc
    if not val:
        raise ValueError(f"Widget parameter '{name}' must not be blank.")
    return val


TABLE_NAME: str = _sanitise_param("table_name", _get_param("table_name"), _SAFE_TABLE_NAME_RE)
# FIX-04: TABLE_TYPE now goes through _sanitise_param() like the other params.
TABLE_TYPE: str = _sanitise_param("table_type", _get_param("table_type"), _SAFE_TABLE_TYPE_RE)
BATCH_DATE: str = _sanitise_param("batch_date", _get_param("batch_date"), _SAFE_DATE_RE)

_LOG.info(
    "Pipeline started",
    extra={
        "stage": "init",
        "table": TABLE_NAME,
        "scd_type": "",
        "batch_date": BATCH_DATE,
        "table_type": TABLE_TYPE,
    },
)


# ── Config Load & Validation ──────────────────────────────────────────────────

# Supported (table_type, scd_type) combinations.
_VALID_COMBOS: set[tuple[str, str]] = {
    ("DIM", "SCD2"),
    ("DIM", "SCD1"),
    ("FACT", "SCD1"),
    ("FACT", "SCD4"),
}


def _load_table_config(table_name: str) -> dict:
    """Load and validate config for *table_name* from F1.config.meta_table_F1.config.

    Args:
        table_name: Logical table name (matches Workflow widget).

    Returns:
        Dict with keys: pk_cols, hash_cols, partition_cols, scd_type,
        dq_rule_group, staging_table, target_table, history_table.

    Raises:
        ValueError: If the table has no config row, or if the
            (table_type, scd_type) combo is unsupported.
    """
    t0 = time.perf_counter()
    rows = (
        spark.table("F1.config.meta_table_config")
        .filter(F.col("table_name") == table_name)
        .limit(1)
        .collect()
    )
    if not rows:
        msg = (
            f"No config found in F1.config.meta_table_config for table '{table_name}'. "
            "Ensure the meta MERGE (setup) has been executed."
        )
        _LOG.error(msg, extra={"stage": "config_load", "table": table_name, "scd_type": "", "batch_date": BATCH_DATE})
        raise ValueError(msg)

    row = rows[0]
    cfg: dict = {
        "pk_cols":        [c.strip() for c in row["pk_cols"].split(",")],
        "hash_cols":      [c.strip() for c in row["hash_cols"].split(",")],
        "partition_cols": [c.strip() for c in row["partition_cols"].split(",")] if row["partition_cols"] else [],
        "scd_type":       row["scd_type"],
        "dq_rule_group":  row["dq_rule_group"],
        "staging_table":  row["staging_table"],
        "target_table":   row["target_table"],
        "history_table":  row["history_table"],
    }

    combo = (TABLE_TYPE, cfg["scd_type"])
    if combo not in _VALID_COMBOS:
        valid_str = ", ".join(f"{t}×{s}" for t, s in sorted(_VALID_COMBOS))
        msg = (
            f"Unsupported (table_type, scd_type) combination: {combo}. "
            f"Valid combinations: {valid_str}."
        )
        _LOG.error(msg, extra={"stage": "config_validate", "table": table_name, "scd_type": cfg["scd_type"], "batch_date": BATCH_DATE})
        raise ValueError(msg)

    _LOG.info(
        "Config loaded",
        extra={
            "stage": "config_load",
            "table": table_name,
            "scd_type": cfg["scd_type"],
            "batch_date": BATCH_DATE,
            "elapsed_s": round(time.perf_counter() - t0, 3),
            **{k: v for k, v in cfg.items() if isinstance(v, str)},
        },
    )
    return cfg


CFG: dict = _load_table_config(TABLE_NAME)
SCD_TYPE: str = CFG["scd_type"]


# ── Data Load ─────────────────────────────────────────────────────────────────

# FIX-07: _load_staging now accepts a batch_date and attempts a partition
# push-down filter on the 'file_date' column when it exists.
# This prevents full-table scans on large Bronze tables with years of history.
# If no 'file_date' column exists the full table is returned unchanged
# (backwards compatible with tables that do not carry this column).
def _load_staging(staging_table: str, batch_date: str) -> DataFrame:
    """Read the Bronze/staging table for the current batch.

    When the staging table contains a ``file_date`` column the read is
    filtered to ``file_date = batch_date`` to avoid full-table scans on
    large partitioned Bronze tables.

    Args:
        staging_table: Fully qualified staging table name.
        batch_date: Processing date string ``yyyy-MM-dd`` used for
            partition push-down when a ``file_date`` column is present.

    Returns:
        Staging :class:`DataFrame`, filtered to the current batch when
        possible.
    """
    t0 = time.perf_counter()
    _LOG.info("Loading staging data", extra={"stage": "load_staging", "source": staging_table})
    df: DataFrame = spark.table(staging_table)

    # Push-down filter: only apply when the column is actually present.
    if "file_date" in df.columns:
        df = df.filter(F.col("file_date") == F.lit(batch_date).cast("date"))
        _LOG.debug(
            "Staging load: file_date push-down applied",
            extra={"stage": "load_staging", "source": staging_table, "batch_date": batch_date},
        )
    else:
        _LOG.warning(
            "Staging table has no 'file_date' column — loading full table. "
            "Consider adding a date partition column to improve performance.",
            extra={"stage": "load_staging", "source": staging_table},
        )

    _LOG.debug(
        "Staging loaded",
        extra={"stage": "load_staging", "source": staging_table, "elapsed_s": round(time.perf_counter() - t0, 3)},
    )
    return df


# ── Bronze → Silver Cast Map ──────────────────────────────────────────────────
# Bronze stores every column as STRING (schema-on-read).  Silver enforces typed
# columns.  Without explicit casts Delta raises DELTA_FAILED_TO_MERGE_FIELDS
# (SQLSTATE 22005) on any typed column (INT, DATE, DOUBLE, FLOAT).
#
# This dict is the single source of truth for all 43 Bronze→Silver promotions,
# derived directly from Tbls_Silver.sql.  Only business columns that differ
# from STRING appear here; audit columns (ingestion_date, row_hash, etc.) are
# added by the pipeline as typed literals and need no cast.

_BRONZE_TO_SILVER_CASTS: dict = {
    "circuits": {
        "circuit_id": "INT",
        "latitude":   "DOUBLE",
        "longitude":  "DOUBLE",
        "altitude":   "INT",
    },
    "races": {
        "race_id":   "INT",
        "race_year": "INT",
        "round":     "INT",
        "circuit_id":"INT",
        "date":      "DATE",
    },
    "constructors": {
        "constructor_id": "INT",
    },
    "drivers": {
        "driver_id": "INT",
        "number":    "INT",
        "dob":       "DATE",
    },
    "results": {
        "result_id":         "INT",
        "race_id":           "INT",
        "driver_id":         "INT",
        "constructor_id":    "INT",
        "number":            "INT",
        "grid":              "INT",
        "position_text":     "INT",
        "position_order":    "INT",
        "points":            "INT",
        "laps":              "INT",
        "milliseconds":      "INT",
        "fastest_lap":       "INT",
        "rank":              "INT",
        "fastest_lap_speed": "FLOAT",
    },
    "pit_stops": {
        "driver_id":    "INT",
        "lap":          "INT",
        "milliseconds": "INT",
        "race_id":      "INT",
        "stop":         "INT",
    },
    "lap_times": {
        "race_id":      "INT",
        "driver_id":    "INT",
        "lap":          "INT",
        "position":     "INT",
        "milliseconds": "INT",
    },
    "qualifying": {
        "constructor_id": "INT",
        "driver_id":      "INT",
        "number":         "INT",
        "position":       "INT",
        "qualify_id":     "INT",
        "race_id":        "INT",
    },
}


def _cast_to_silver(df: DataFrame, table_name: str) -> DataFrame:
    """Cast Bronze STRING columns to their Silver target types.

    Bronze tables land every column as STRING.  Silver tables enforce typed
    schemas (INT, DATE, DOUBLE, FLOAT).  This function applies the casts
    defined in ``_BRONZE_TO_SILVER_CASTS`` so that every downstream write —
    overwrite, append, or Delta MERGE — sees columns whose Spark types exactly
    match the registered Silver Delta schema.

    Only columns listed in ``_BRONZE_TO_SILVER_CASTS[table_name]`` are touched;
    all others pass through unchanged.  Columns listed in the cast map but
    absent from *df* are skipped with a WARNING so the pipeline is backwards-
    compatible if a Bronze source ever drops a column.

    Must be called BEFORE ``with_hash()`` so the hash is computed over typed
    values, and BEFORE ``run_dq_checks()`` so numeric range rules evaluate on
    the correct types.

    Args:
        df: Raw Bronze DataFrame (all business columns as STRING).
        table_name: Logical table name used to look up ``_BRONZE_TO_SILVER_CASTS``.

    Returns:
        DataFrame with business columns type-promoted to match the Silver schema.
    """
    t0 = time.perf_counter()
    cast_map: dict = _BRONZE_TO_SILVER_CASTS.get(table_name.lower(), {})

    if not cast_map:
        _LOG.debug(
            "No Bronze→Silver casts required",
            extra={"stage": "cast", "table": table_name},
        )
        return df

    cast_applied: List[str] = []
    for col_name, target_type in cast_map.items():
        if col_name in df.columns:
            df = df.withColumn(col_name, F.col(col_name).cast(target_type))
            cast_applied.append(f"{col_name}→{target_type}")
        else:
            _LOG.warning(
                "Cast column absent from staging DataFrame — skipping",
                extra={
                    "stage": "cast",
                    "table": table_name,
                    "missing_col": col_name,
                    "expected_type": target_type,
                },
            )

    _LOG.debug(
        "Bronze→Silver casts applied",
        extra={
            "stage":        "cast",
            "table":        table_name,
            "columns_cast": len(cast_applied),
            "detail":       ", ".join(cast_applied),
            "elapsed_s":    round(time.perf_counter() - t0, 3),
        },
    )
    return df


# ── DQ Framework ──────────────────────────────────────────────────────────────

# FIX-03: Allowlist of SQL token patterns considered safe for DQ expressions.
# spark_sql_expr is stored in F1.config.meta_dq_rules and interpolated into
# df.filter(). Anyone with WRITE access to that table could inject arbitrary
# Spark SQL.  We validate the raw expression against this allowlist before
# passing it to Spark.  The allowlist covers all rule_types used in the
# production metadata (NOT_NULL, RANGE, IN_LIST, EXPR) while blocking
# dangerous patterns (subqueries, UDFs, DDL keywords, semicolons, etc.).
_DQ_EXPR_ALLOWLIST_RE: re.Pattern = re.compile(
    r"^[\w\s\(\)\.\,\'\"\=\!\<\>\+\-\*\/\%\|&\^\~\[\]]+$",
    re.IGNORECASE,
)
_DQ_EXPR_BLOCKLIST_RE: re.Pattern = re.compile(
    r"\b(SELECT|INSERT|UPDATE|DELETE|DROP|CREATE|ALTER|EXEC|EXECUTE|CALL"
    r"|TRUNCATE|MERGE|GRANT|REVOKE|LOAD|COPY|PUT|GET)\b"
    r"|--|;|/\*|\*/",
    re.IGNORECASE,
)


def _validate_dq_expr(rule_id: str, expr: str) -> str:
    """Validate a DQ SQL expression against the safety allowlist/blocklist.

    Args:
        rule_id: Rule identifier (for error messages).
        expr: Raw ``spark_sql_expr`` value from ``F1.config.meta_dq_rules``.

    Returns:
        The original *expr* if it passes both checks.

    Raises:
        ValueError: If *expr* contains blocked SQL tokens or characters
            outside the safe allowlist.
    """
    if _DQ_EXPR_BLOCKLIST_RE.search(expr):
        raise ValueError(
            f"DQ rule '{rule_id}' spark_sql_expr contains a blocked SQL token. "
            f"Expression: {expr!r}. "
            "Check F1.config.meta_dq_rules for tampering."
        )
    if not _DQ_EXPR_ALLOWLIST_RE.fullmatch(expr.strip()):
        raise ValueError(
            f"DQ rule '{rule_id}' spark_sql_expr contains characters outside "
            f"the safe allowlist. Expression: {expr!r}. "
            "Check F1.config.meta_dq_rules for tampering."
        )
    return expr


def run_dq_checks(
    df: DataFrame,
    cfg: dict,
    batch_date: str,
) -> DataFrame:
    """Execute all DQ rules for the table in a single parallel Spark job.

    Steps:
      1. Collect all rule rows for the table's ``dq_rule_group`` (small metadata).
      2. Validate each ``spark_sql_expr`` against the safety allowlist (FIX-03).
      3. Build one violation DataFrame per rule (lazy; no actions yet).
      4. Union all violation DataFrames in a single ``reduce`` → one Spark job.
      5. Cache violations; count by severity (single scan).
      6. Write violations to ``F1.config.dq_violations`` (append).
      7. Log WARNs; raise ``RuntimeError`` for ERROR + FAIL_PIPELINE.
      8. Return clean records via left-anti join on PKs.

    Args:
        df: Staged input DataFrame.
        cfg: Table config dict (requires ``pk_cols``, ``dq_rule_group``).
        batch_date: Processing date string ``yyyy-MM-dd``.

    Returns:
        DataFrame containing only rows that passed all DQ checks.

    Raises:
        ValueError: If any DQ expression fails the safety allowlist check.
        RuntimeError: If any ERROR-severity / FAIL_PIPELINE rule is violated.
    """
    t0 = time.perf_counter()
    rule_group: str = cfg["dq_rule_group"]
    pk_cols: List[str] = cfg["pk_cols"]

    _LOG.info("DQ checks starting", extra={"stage": "dq", "rule_group": rule_group})

    # 1. Collect rule metadata (always small — safe to collect).
    rules: list = (
        spark.table("F1.config.meta_dq_rules")
        .filter(F.col("rule_group") == rule_group)
        .collect()
    )

    if not rules:
        _LOG.info("No DQ rules found — skipping", extra={"stage": "dq", "rule_group": rule_group})
        return df

    # 2. Build one violation DataFrame per rule (lazy).
    pk_concat_expr = F.to_json(F.struct(*[F.col(c) for c in pk_cols]))
    row_json_expr = F.to_json(F.struct("*"))

    violation_dfs: List[DataFrame] = []
    for rule in rules:
        # FIX-03: Validate expression before handing it to Spark.
        expr: str = _validate_dq_expr(rule["rule_id"], rule["spark_sql_expr"])
        violations = (
            df.filter(f"NOT ({expr})")
            .select(
                F.lit(TABLE_NAME).alias("table_name"),
                F.lit(batch_date).cast("date").alias("batch_date"),
                F.lit(rule_group).alias("rule_group"),
                F.lit(rule["rule_id"]).alias("rule_id"),
                F.lit(rule["rule_type"]).alias("rule_type"),
                F.lit(rule["column_name"]).alias("column_name"),
                F.lit(rule["severity"]).alias("dq_severity"),
                F.lit(rule["reject_action"]).alias("reject_action"),
                pk_concat_expr.alias("pk_values"),
                row_json_expr.alias("raw_row"),
                F.current_timestamp().alias("logged_at"),
            )
        )
        violation_dfs.append(violations)

    # 3. Union all violation DataFrames — ONE Spark job, not N.
    all_violations: DataFrame = reduce(DataFrame.unionByName, violation_dfs)

    # Materialise violations once so the subsequent write and count action
    # do not each trigger a full re-scan of the (potentially large) lazy plan.
    all_violations.cache()

    # 4. Count violations grouped by severity BEFORE writing (single scan).
    sev_counts: dict = {
        row["dq_severity"]: row["cnt"]
        for row in all_violations.groupBy("dq_severity").count().withColumnRenamed("count", "cnt").collect()
    }

    warn_count: int = sev_counts.get("WARN", 0)
    error_count: int = sev_counts.get("ERROR", 0)

    # 5. Write violations to audit table via the single write-path helper.
    _write_delta(all_violations, "F1.config.dq_violations", mode="append", partition_cols=[])

    # 6. Log and raise as appropriate.
    if warn_count:
        _LOG.warning(
            "DQ WARN violations found",
            extra={"stage": "dq", "warn_violations": warn_count},
        )

    if error_count:
        # Check reject_action — only FAIL_PIPELINE stops the run.
        fail_count: int = (
            all_violations
            .filter((F.col("dq_severity") == "ERROR") & (F.col("reject_action") == "FAIL_PIPELINE"))
            .limit(1)
            .count()
        )
        _LOG.error(
            "DQ ERROR violations found",
            extra={"stage": "dq", "error_violations": error_count, "fail_pipeline": fail_count > 0},
        )
        if fail_count:
            raise RuntimeError(
                f"DQ check failed for table '{TABLE_NAME}': "
                f"{error_count} ERROR violation(s) with FAIL_PIPELINE action. "
                "Check F1.config.dq_violations for details."
            )

    # 7. Return clean records (left-anti join on PKs — stays distributed).
    violation_pks: DataFrame = (
        all_violations
        .filter(F.col("reject_action") != "IGNORE")
        .select(F.from_json(F.col("pk_values"), df.select(*pk_cols).schema).alias("pks"))
        .select([F.col(f"pks.{c}").alias(c) for c in pk_cols])
        .distinct()
    )

    clean_df: DataFrame = df.join(violation_pks, on=pk_cols, how="left_anti")

    # Release cached violations from memory — no longer needed after the join.
    all_violations.unpersist()

    _LOG.info(
        "DQ checks complete",
        extra={
            "stage": "dq",
            "warn_violations": warn_count,
            "error_violations": error_count,
            "elapsed_s": round(time.perf_counter() - t0, 3),
        },
    )
    return clean_df


# ── Helpers ───────────────────────────────────────────────────────────────────

def with_hash(df: DataFrame, hash_cols: List[str], alias: str = "row_hash") -> DataFrame:
    """Append a SHA-256 hash column over *hash_cols* (NULL-safe).

    ``COALESCE(col, '__NULL__')`` is applied per column before concatenation
    so that a NULL in any single column does not collapse the entire hash.

    Args:
        df: Input DataFrame.
        hash_cols: Ordered list of column names to include in the hash.
        alias: Name of the resulting hash column (default ``row_hash``).

    Returns:
        DataFrame with the new hash column appended.
    """
    concat_expr = F.concat_ws(
        "||",
        *[F.coalesce(F.col(c).cast(StringType()), F.lit("__NULL__")) for c in hash_cols],
    )
    return df.withColumn(alias, F.sha2(concat_expr, 256))


def _write_delta(
    df: DataFrame,
    target: str,
    mode: str,
    partition_cols: List[str],
    overwrite_schema: bool = False,
) -> None:
    """Unified Delta write helper — all writes must go through this function.

    Args:
        df: DataFrame to write.
        target: Fully qualified target table name.
        mode: Spark write mode (``"overwrite"`` or ``"append"``).
        partition_cols: Columns to partition by; empty list skips partitioning.
        overwrite_schema: Set ``True`` only on initial/overwrite loads.

    Side effects:
        Writes data to the Delta table at *target*.
    """
    t0 = time.perf_counter()
    writer = df.write.format("delta").mode(mode)
    #if overwrite_schema:
        #writer = writer.option("overwriteSchema", "true")
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.saveAsTable(target)
    _LOG.debug(
        "Delta write complete",
        extra={
            "stage": "write",
            "target": target,
            "mode": mode,
            "partition_cols": partition_cols,
            "elapsed_s": round(time.perf_counter() - t0, 3),
        },
    )


# ── SCD1 Merge ────────────────────────────────────────────────────────────────

def _scd1_merge(
    source_df: DataFrame,
    target_table: str,
    pk_cols: List[str],
    hash_cols: List[str],
    partition_cols: List[str],
    batch_date: str,
) -> Optional[int]:
    """Hash-based SCD1 upsert into a Delta target table.

    - **Initial load** (target empty or absent): full overwrite with schema.
    - **Incremental**: Delta MERGE on *pk_cols*.
      - MATCHED + hash changed → UPDATE all columns + ``updated_at``.
      - NOT MATCHED → INSERT full row.

    Args:
        source_df: Incoming (already DQ-clean, hashed) DataFrame.
        target_table: Fully qualified Silver target table name.
        pk_cols: Primary-key column list (used for MERGE condition).
        hash_cols: Columns included in ``row_hash`` (already computed).
        partition_cols: Partition columns for writes.
        batch_date: Processing date string ``yyyy-MM-dd``.

    Returns:
        Number of rows written on initial load; ``None`` on incremental MERGE
        (Delta MERGE metrics are not directly exposed via the Python API).

    Side effects:
        Modifies the Delta table at *target_table*.
    """
    t0 = time.perf_counter()
    _LOG.info("SCD1 merge starting", extra={"stage": "scd1", "target": target_table})

    # Decorate source with audit columns.
    src = (
        source_df
        .withColumn("ingestion_date", F.current_timestamp())
        .withColumn("data_source", F.lit(TABLE_NAME))
        .withColumn("file_date", F.lit(batch_date))
        .withColumn("eff_start_date", F.lit(batch_date).cast("date"))
        .withColumn("eff_end_date", F.lit("9999-12-31").cast("date"))
        .withColumn("is_current", F.lit(1))
        .withColumn("created_at", F.current_timestamp())
        .withColumn("updated_at", F.current_timestamp())
    )

    # Initial load guard — spark.catalog.tableExists() is Unity Catalog-safe.
    target_empty: bool = (
        not spark.catalog.tableExists(target_table)
        or spark.table(target_table).limit(1).count() == 0
    )

    if target_empty:
        _LOG.info("Initial load — overwriting target", extra={"stage": "scd1", "target": target_table})
        # Count BEFORE write to avoid a second post-write scan.
        rows_written: int = src.count()
        _write_delta(src, target_table, mode="overwrite", partition_cols=partition_cols, overwrite_schema=True)
        _LOG.info(
            "SCD1 initial load complete",
            extra={"stage": "scd1", "rows_written": rows_written, "elapsed_s": round(time.perf_counter() - t0, 3)},
        )
        return rows_written

    # Incremental MERGE.
    delta_tgt = DeltaTable.forName(spark, target_table)

    join_condition: str = " AND ".join([f"tgt.{c} = src.{c}" for c in pk_cols])

    # All non-PK columns become update/insert targets.
    all_cols: List[str] = [c for c in src.columns if c not in pk_cols and c != "created_at"]
    update_map: dict = {c: f"src.{c}" for c in all_cols}
    insert_map: dict = {c: f"src.{c}" for c in src.columns}

    (
        delta_tgt.alias("tgt")
        .merge(src.alias("src"), join_condition)
        .whenMatchedUpdate(
            condition="tgt.row_hash <> src.row_hash",
            set={**update_map, "updated_at": "current_timestamp()"},
        )
        .whenNotMatchedInsert(values=insert_map)
        .execute()
    )

    elapsed: float = round(time.perf_counter() - t0, 3)
    _LOG.info(
        "SCD1 merge complete",
        # FIX-05: Return None (not 0) for incremental path — Delta MERGE row
        # counts are not exposed via the Python API.  Callers should treat
        # None as "count unavailable" rather than "zero rows written".
        extra={"stage": "scd1", "target": target_table, "rows_merged": "N/A (MERGE)", "elapsed_s": elapsed},
    )
    # FIX-05: Return None instead of misleading sentinel 0.
    return None


# ── SCD2 ──────────────────────────────────────────────────────────────────────

def run_scd2(
    source_df: DataFrame,
    target_table: str,
    pk_cols: List[str],
    hash_cols: List[str],
    partition_cols: List[str],
    batch_date: str,
) -> None:
    """Hash-based SCD2 processing for DIM tables.

    Algorithm:
      1. Compute ``row_hash`` over *hash_cols*.
      2. If target is empty → initial load (overwrite, all rows open-dated).
      3. Otherwise:
         - Outer-join source vs active target (``is_current = 1``) on *pk_cols*.
         - **NEW** rows (no target match): insert with open dates.
         - **CHANGED** rows (hash mismatch): expire via MERGE then append new version.
         - **UNCHANGED** rows (hash match): skip — no Spark action.
      4. No-op guard: return early if both new and changed sets are empty.

    Args:
        source_df: DQ-clean incoming DataFrame.
        target_table: Fully qualified Silver DIM table.
        pk_cols: Primary-key columns.
        hash_cols: Columns included in the hash.
        partition_cols: Partition columns for writes.
        batch_date: Processing date string ``yyyy-MM-dd``.

    Side effects:
        Modifies the Delta table at *target_table*.
    """
    t0 = time.perf_counter()
    _LOG.info("SCD2 starting", extra={"stage": "scd2", "target": target_table})

    # source_df arrives pre-hashed from the pipeline (with_hash already applied
    # upstream before DQ checks).  Do NOT call with_hash() again here.
    src: DataFrame = source_df

    # Initial load guard — spark.catalog.tableExists() is Unity Catalog-safe.
    target_empty: bool = (
        not spark.catalog.tableExists(target_table)
        or spark.table(target_table).limit(1).count() == 0
    )

    if target_empty:
        _LOG.info("SCD2 initial load", extra={"stage": "scd2", "target": target_table})
        initial = (
            src
            .withColumn("eff_start_date", F.lit(batch_date).cast("date"))
            .withColumn("eff_end_date", F.lit("9999-12-31").cast("date"))
            .withColumn("is_current", F.lit(1))
            .withColumn("ingestion_date", F.current_timestamp())
            .withColumn("data_source", F.lit(TABLE_NAME))
            .withColumn("file_date", F.lit(batch_date))
            .withColumn("created_at", F.current_timestamp())
            .withColumn("updated_at", F.current_timestamp())
        )
        # Count BEFORE write to avoid a second post-write scan.
        rows_written: int = initial.count()
        _write_delta(initial, target_table, mode="overwrite", partition_cols=partition_cols, overwrite_schema=True)
        _LOG.info(
            "SCD2 initial load complete",
            extra={"stage": "scd2", "rows_written": rows_written, "elapsed_s": round(time.perf_counter() - t0, 3)},
        )
        return

    # Active target records.
    active_tgt: DataFrame = spark.table(target_table).filter(F.col("is_current") == 1)

    # Outer join — prefix source columns to avoid ambiguity.
    src_prefixed = src.select([F.col(c).alias(f"src_{c}") for c in src.columns])
    tgt_prefixed = active_tgt.select(
        *[F.col(c).alias(f"tgt_{c}") for c in pk_cols],
        F.col("row_hash").alias("tgt_row_hash"),
    )

    join_cond = reduce(
        lambda a, b: a & b,
        [src_prefixed[f"src_{c}"] == tgt_prefixed[f"tgt_{c}"] for c in pk_cols],
    )
    joined: DataFrame = src_prefixed.join(tgt_prefixed, on=join_cond, how="left")

    # NEW rows — no target match.
    new_rows: DataFrame = joined.filter(tgt_prefixed[f"tgt_{pk_cols[0]}"].isNull()).select(
        [F.col(f"src_{c}").alias(c) for c in src.columns]
    )

    # CHANGED rows — hash mismatch.
    changed_rows: DataFrame = joined.filter(
        tgt_prefixed[f"tgt_{pk_cols[0]}"].isNotNull()
        & (F.col("src_row_hash") != F.col("tgt_row_hash"))
    ).select([F.col(f"src_{c}").alias(c) for c in src.columns])

    # FIX-09: Cache changed_rows before MERGE + count to avoid re-scanning
    # the outer join twice (once for the MERGE input, once for expired_count).
    changed_rows.cache()

    # No-op guard — cheap limit(1).count() instead of .isEmpty() on large DFs.
    new_empty: bool = new_rows.limit(1).count() == 0
    changed_empty: bool = changed_rows.limit(1).count() == 0

    if new_empty and changed_empty:
        changed_rows.unpersist()
        _LOG.info(
            "SCD2 no-op — no new or changed rows",
            extra={"stage": "scd2", "target": target_table, "elapsed_s": round(time.perf_counter() - t0, 3)},
        )
        return

    delta_tgt = DeltaTable.forName(spark, target_table)
    join_cond_sql: str = " AND ".join([f"tgt.{c} = src.{c}" for c in pk_cols])

    # Expire changed records.
    if not changed_empty:
        (
            delta_tgt.alias("tgt")
            .merge(changed_rows.alias("src"), join_cond_sql + " AND tgt.is_current = 1")
            .whenMatchedUpdate(
                set={
                    "is_current": "0",
                    "eff_end_date": f"'{batch_date}'",
                    "updated_at": "current_timestamp()",
                }
            )
            .execute()
        )
        # FIX-09: changed_rows is cached — this count hits memory, not the join.
        expired_count: int = changed_rows.count()
        _LOG.debug("SCD2 expired rows", extra={"stage": "scd2", "expired_rows": expired_count})
    else:
        expired_count = 0

    # Append new versions for new + changed rows.
    def _tag_new(df: DataFrame) -> DataFrame:
        """Decorate a DataFrame of new/changed rows with SCD2 open-date audit columns.

        Args:
            df: DataFrame of rows to insert (new or changed versions).

        Returns:
            DataFrame with SCD2 audit columns added or overwritten.
        """
        return (
            df
            .withColumn("eff_start_date", F.lit(batch_date).cast("date"))
            .withColumn("eff_end_date", F.lit("9999-12-31").cast("date"))
            .withColumn("is_current", F.lit(1))
            .withColumn("ingestion_date", F.current_timestamp())
            .withColumn("data_source", F.lit(TABLE_NAME))
            .withColumn("file_date", F.lit(batch_date))
            .withColumn("created_at", F.current_timestamp())
            .withColumn("updated_at", F.current_timestamp())
        )

    # Build the insert set: new records + new versions of changed records.
    if not new_empty and not changed_empty:
        inserts: DataFrame = new_rows.unionByName(changed_rows)
    elif not new_empty:
        inserts = new_rows
    else:
        inserts = changed_rows

    # Count BEFORE write to avoid a post-write re-scan for logging.
    new_count: int = 0 if new_empty else new_rows.count()
    changed_count: int = 0 if changed_empty else changed_rows.count()

    _write_delta(_tag_new(inserts), target_table, mode="append", partition_cols=partition_cols)

    # FIX-09: Release cached changed_rows after the write.
    changed_rows.unpersist()

    _LOG.info(
        "SCD2 complete",
        extra={
            "stage": "scd2",
            "new_rows": new_count,
            "changed_rows": changed_count,
            "expired_rows": expired_count,
            "elapsed_s": round(time.perf_counter() - t0, 3),
        },
    )


# ── SCD4 ──────────────────────────────────────────────────────────────────────

def run_scd4(
    source_df: DataFrame,
    target_table: str,
    history_table: str,
    pk_cols: List[str],
    hash_cols: List[str],
    partition_cols: List[str],
    batch_date: str,
) -> None:
    """SCD4: update current snapshot (SCD1) then append snapshot to history.

    Steps:
      1. Call ``_scd1_merge()`` to update the current FACT snapshot.
      2. Read the current snapshot (batch-touched rows only) and append to
         *history_table* with ``archived_at`` audit column.

    Args:
        source_df: DQ-clean incoming DataFrame (pre-hashed).
        target_table: Current-snapshot Silver FACT table.
        history_table: SCD4 history table (``{target}_history``).
        pk_cols: Primary-key columns.
        hash_cols: Hash columns.
        partition_cols: Partition columns for writes.
        batch_date: Processing date string ``yyyy-MM-dd``.

    Side effects:
        Modifies *target_table* (via SCD1) and appends to *history_table*.
    """
    t0 = time.perf_counter()
    _LOG.info("SCD4 starting", extra={"stage": "scd4", "target": target_table, "history": history_table})

    # 1. Update current snapshot via SCD1.
    _scd1_merge(
        source_df=source_df,
        target_table=target_table,
        pk_cols=pk_cols,
        hash_cols=hash_cols,
        partition_cols=partition_cols,
        batch_date=batch_date,
    )

    # 2. Append only the rows that were touched in this batch to the history table.
    #    Joining the current snapshot back against source PKs ensures we record
    #    exactly the new/updated versions written by _scd1_merge, not the entire
    #    table (which would duplicate all pre-existing rows every run).
    snapshot: DataFrame = (
        spark.table(target_table)
        .join(
            source_df.select(*pk_cols).distinct(),
            on=pk_cols,
            how="inner",
        )
        .withColumn("archived_at", F.current_timestamp())
    )

    # Count BEFORE write to avoid a post-write re-scan for logging.
    hist_rows: int = snapshot.count()
    _write_delta(snapshot, history_table, mode="append", partition_cols=partition_cols)

    _LOG.info(
        "SCD4 history append complete",
        extra={
            "stage": "scd4",
            "history_table": history_table,
            "rows_appended": hist_rows,
            "elapsed_s": round(time.perf_counter() - t0, 3),
        },
    )


# ── Pipeline Execution ────────────────────────────────────────────────────────

with _LogCtx(table=TABLE_NAME, scd_type=SCD_TYPE, batch_date=BATCH_DATE):

    pipeline_t0 = time.perf_counter()

    # Load staging data.
    # FIX-07: Pass BATCH_DATE so _load_staging can apply partition push-down.
    staging_df: DataFrame = _load_staging(CFG["staging_table"], BATCH_DATE)

    # Cast Bronze STRING columns to Silver target types (INT, DATE, DOUBLE, FLOAT).
    # Must run BEFORE with_hash() so the hash is computed over typed values,
    # and BEFORE run_dq_checks() so numeric range rules evaluate correctly.
    staging_df = _cast_to_silver(staging_df, TABLE_NAME)

    # Pre-compute row_hash for DQ-clean path (needed by all SCD handlers).
    staged_hashed: DataFrame = with_hash(staging_df, CFG["hash_cols"], alias="row_hash")

    # Run DQ checks.
    clean_df: DataFrame = run_dq_checks(staged_hashed, CFG, BATCH_DATE)

    # FIX-08: Cache clean_df before SCD routing.
    # SCD4 calls _scd1_merge (reads clean_df) then re-uses source PKs for the
    # history join — both re-evaluate the DQ anti-join lineage unless cached.
    # SCD1 and SCD2 also benefit from caching on large datasets.
    clean_df.cache()

    # ── Routing ──────────────────────────────────────────────────────────────
    if SCD_TYPE == "SCD2":
        run_scd2(
            source_df=clean_df,
            target_table=CFG["target_table"],
            pk_cols=CFG["pk_cols"],
            hash_cols=CFG["hash_cols"],
            partition_cols=CFG["partition_cols"],
            batch_date=BATCH_DATE,
        )

    elif SCD_TYPE == "SCD1":
        _scd1_merge(
            source_df=clean_df,
            target_table=CFG["target_table"],
            pk_cols=CFG["pk_cols"],
            hash_cols=CFG["hash_cols"],
            partition_cols=CFG["partition_cols"],
            batch_date=BATCH_DATE,
        )

    elif SCD_TYPE == "SCD4":
        if not CFG.get("history_table"):
            raise ValueError(
                f"SCD4 requires 'history_table' in meta_table_config for table '{TABLE_NAME}'."
            )
        run_scd4(
            source_df=clean_df,
            target_table=CFG["target_table"],
            history_table=CFG["history_table"],
            pk_cols=CFG["pk_cols"],
            hash_cols=CFG["hash_cols"],
            partition_cols=CFG["partition_cols"],
            batch_date=BATCH_DATE,
        )

    else:
        # Should never reach here due to config validation above.
        raise ValueError(f"Unhandled SCD type: {SCD_TYPE}")

    # FIX-08: Release clean_df from cache — all SCD handlers have completed.
    clean_df.unpersist()

    # ── Update ETL batch status ───────────────────────────────────────────────
    # FIX-06: Wrapped in try/except so a missing status table or transient
    # write failure raises a clear, logged error.  Without this guard, a
    # failure here would surface as an unhandled crash *after* Silver data
    # has already been written, leaving the batch with no success record and
    # no actionable error message.
    try:
        spark.sql(f"""
            MERGE INTO F1.Silver.ETL_batch_status AS tgt
            USING (SELECT
                     '{TABLE_NAME}'          AS table_name,
                     DATE('{BATCH_DATE}')    AS batch_date,
                     'SUCCESS'               AS status,
                     current_timestamp()     AS updated_at
                  ) AS src
            ON tgt.table_name = src.table_name AND tgt.batch_date = src.batch_date
            WHEN MATCHED THEN UPDATE SET
                tgt.status     = src.status,
                tgt.updated_at = src.updated_at
            WHEN NOT MATCHED THEN INSERT *
        """)
    except Exception as status_exc:
        # Data is already written — log the failure clearly so ops teams can
        # backfill the status record manually without re-running the pipeline.
        _LOG.error(
            "ETL_batch_status update failed — Silver data was written successfully "
            "but the status record could not be upserted. Manual backfill required.",
            extra={
                "stage": "batch_status",
                "table": TABLE_NAME,
                "batch_date": BATCH_DATE,
                "error": str(status_exc),
            },
        )
        raise RuntimeError(
            f"ETL_batch_status MERGE failed for table '{TABLE_NAME}' "
            f"batch '{BATCH_DATE}'. Silver data IS written. "
            "Backfill the status record and investigate the error above."
        ) from status_exc

    _LOG.info(
        "Pipeline complete",
        extra={
            "stage": "pipeline_end",
            "total_elapsed_s": round(time.perf_counter() - pipeline_t0, 3),
        },
    )
